# Issue with Rooki workflow serialization tree reuse

This notebook documents a fixed workflow serialization issue reported by a user. The problem was caused by mutable default arguments in `Input._tree()` and `Operator._tree()`: separate calls to `_serialise()` could reuse the same workflow tree and accidentally carry previous `inputs` and `steps` into a later request.

The example below builds two independent workflows in the same Python process. The second workflow should contain only `tas`, not the earlier `huss` request.

## Imports

The notebook serializes workflows to show the fixed client-side behavior. Importing `rooki.operators` initializes the Rooki WPS client and may contact the configured service, for example `rook.dkrz.de`.

In [1]:
import json
from pprint import pprint

from rooki import operators as rookops

## First workflow

In [2]:
workflow_huss = rookops.Subset(
    rookops.Input("huss", ["c3s-cordex.example.huss"]),
    time="2006/2006",
    time_components="month:jan|year:2006",
)

doc_huss = json.loads(workflow_huss._serialise())
pprint(doc_huss)

{'doc': 'workflow',
 'inputs': {'huss': ['c3s-cordex.example.huss']},
 'outputs': {'output': 'subset_huss_1/output'},
 'steps': {'subset_huss_1': {'in': {'collection': 'inputs/huss',
                                    'time': '2006/2006',
                                    'time_components': 'month:jan|year:2006'},
                             'run': 'subset'}}}


## Second workflow in the same process

In [3]:
workflow_tas = rookops.Subset(
    rookops.Input("tas", ["c3s-cordex.example.tas"]),
    time="2007/2007",
    time_components="month:feb|year:2007",
)

doc_tas = json.loads(workflow_tas._serialise())
pprint(doc_tas)

{'doc': 'workflow',
 'inputs': {'tas': ['c3s-cordex.example.tas']},
 'outputs': {'output': 'subset_tas_1/output'},
 'steps': {'subset_tas_1': {'in': {'collection': 'inputs/tas',
                                   'time': '2007/2007',
                                   'time_components': 'month:feb|year:2007'},
                            'run': 'subset'}}}


## Check the fixed behavior

The second serialized workflow must be isolated from the first one.

In [4]:
assert set(doc_huss["inputs"]) == {"huss"}
assert set(doc_huss["steps"]) == {"subset_huss_1"}

assert set(doc_tas["inputs"]) == {"tas"}
assert set(doc_tas["steps"]) == {"subset_tas_1"}
assert "huss" not in doc_tas["inputs"]
assert "subset_huss_1" not in doc_tas["steps"]

print("OK: each _serialise() call uses a fresh workflow tree.")

OK: each _serialise() call uses a fresh workflow tree.


## Input-only serialization

The same isolation also applies when serializing bare `Input` objects.

In [5]:
input_huss = json.loads(rookops.Input("huss", ["c3s-cordex.example.huss"])._serialise())
input_tas = json.loads(rookops.Input("tas", ["c3s-cordex.example.tas"])._serialise())

assert input_huss["inputs"] == {"huss": ["c3s-cordex.example.huss"]}
assert input_tas["inputs"] == {"tas": ["c3s-cordex.example.tas"]}

pprint(input_huss)
pprint(input_tas)

{'doc': 'workflow', 'inputs': {'huss': ['c3s-cordex.example.huss']}}
{'doc': 'workflow', 'inputs': {'tas': ['c3s-cordex.example.tas']}}
